# Barter-RS: Engine & Backtesting System

This notebook demonstrates how to set up the full Barter trading engine for
backtesting with historical market data and mock execution.

## Topics Covered
- Loading `SystemConfig` from JSON
- Building `IndexedInstruments` from config
- Using `HistoricalClock` for time simulation
- Configuring `SystemBuilder` with strategy and risk manager
- Running the engine and generating a `TradingSummary`
- Engine lifecycle: enable trading, cancel orders, close positions, shutdown

In [ ]:
:dep barter = { version = "0.12" }
:dep barter-data = { version = "0.11" }
:dep barter-instrument = { version = "0.3" }
:dep barter-integration = { version = "0.10" }
:dep tokio = { version = "1", features = ["full"] }
:dep futures = "0.3"
:dep serde_json = "1.0"
:dep rust_decimal = { version = "1.36", features = ["maths"] }
:dep rust_decimal_macros = "1.29"
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter", "json"] }

## 1. System Configuration

Barter uses a JSON-based `SystemConfig` that defines:
- **Instruments**: which exchange/asset pairs to trade
- **Executions**: mock or live execution clients with initial balances

Here's the structure of a typical config:

In [ ]:
let config_json = r#"{
  "instruments": [
    {
      "exchange": "binance_spot",
      "name_exchange": "BTCUSDT",
      "underlying": { "base": "btc", "quote": "usdt" },
      "quote": "underlying_quote",
      "kind": "spot"
    },
    {
      "exchange": "binance_spot",
      "name_exchange": "ETHUSDT",
      "underlying": { "base": "eth", "quote": "usdt" },
      "quote": "underlying_quote",
      "kind": "spot"
    }
  ],
  "executions": [
    {
      "mocked_exchange": "binance_spot",
      "latency_ms": 100,
      "fees_percent": 0.05,
      "initial_state": {
        "exchange": "binance_spot",
        "balances": [
          { "asset": "usdt", "balance": { "total": 10000, "free": 10000 }, "time_exchange": "2025-01-01T00:00:00Z" },
          { "asset": "btc", "balance": { "total": 0.1, "free": 0.1 }, "time_exchange": "2025-01-01T00:00:00Z" },
          { "asset": "eth", "balance": { "total": 1.0, "free": 1.0 }, "time_exchange": "2025-01-01T00:00:00Z" }
        ],
        "instruments": [
          { "instrument": "BTCUSDT", "orders": [] },
          { "instrument": "ETHUSDT", "orders": [] }
        ]
      }
    }
  ]
}"#;

println!("Config JSON defined with 2 instruments and mock execution.");
println!("Starting balance: 10,000 USDT + 0.1 BTC + 1.0 ETH");

## 2. Building the System

The `SystemBuilder` wires together all components:
- **Clock**: `LiveClock` for real-time, `HistoricalClock` for backtesting
- **Strategy**: implements `AlgoStrategy` + `ClosePositionsStrategy` + `OnDisconnectStrategy`
- **RiskManager**: filters/approves orders before execution
- **EngineFeedMode**: `Iterator` (sync) or `Stream` (async)
- **AuditMode**: `Enabled` to replicate state, `Disabled` for performance

In [ ]:
use barter::{
    EngineEvent,
    engine::{
        clock::HistoricalClock,
        state::{
            global::DefaultGlobalData,
            instrument::{data::DefaultInstrumentMarketData, filter::InstrumentFilter},
            trading::TradingState,
        },
    },
    risk::DefaultRiskManager,
    statistic::time::Daily,
    strategy::DefaultStrategy,
    system::{
        builder::{AuditMode, EngineFeedMode, SystemArgs, SystemBuilder},
        config::SystemConfig,
    },
};
use barter_data::streams::{
    consumer::{MarketStreamEvent, MarketStreamResult},
    reconnect::{Event, stream::ReconnectingStream},
};
use barter_data::event::DataKind;
use barter_instrument::{index::IndexedInstruments, instrument::InstrumentIndex};
use futures::{Stream, StreamExt, stream};
use rust_decimal_macros::dec;
use std::time::Duration;

// Parse the config
let config: SystemConfig = serde_json::from_str(config_json)
    .expect("failed to parse SystemConfig");

let instruments = IndexedInstruments::new(config.instruments);
println!("Loaded {} instruments from config", instruments.instruments().len());

// Note: In a real backtest, you would load historical market data from a file or database.
// Here we create an empty stream for demonstration.
let empty_events: Vec<MarketStreamResult<InstrumentIndex, DataKind>> = vec![];
let clock = HistoricalClock::new(chrono::Utc::now());
let market_stream = stream::iter(empty_events)
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

println!("Historical clock and market stream prepared.");

## 3. System Lifecycle

The typical lifecycle of a Barter system:

```
SystemArgs::new(...)          // Wire components
  → SystemBuilder::new(args)  // Configure options
  → .build()?                 // Validate and construct
  → .init_with_runtime(...)   // Spawn async tasks
  → system.trading_state(Enabled)  // Start trading
  → ... run ...
  → system.cancel_orders(...)      // Pre-shutdown
  → system.close_positions(...)    // Flatten positions
  → system.shutdown()              // Clean shutdown
  → engine.trading_summary_generator(rfr).generate(Daily)
```

In [ ]:
// Construct SystemArgs
let args = SystemArgs::new(
    &instruments,
    config.executions,
    clock,
    DefaultStrategy::default(),
    DefaultRiskManager::default(),
    market_stream,
    DefaultGlobalData::default(),
    |_| DefaultInstrumentMarketData::default(),
);

// Build the system
let system = SystemBuilder::new(args)
    .engine_feed_mode(EngineFeedMode::Stream)
    .audit_mode(AuditMode::Disabled)
    .trading_state(TradingState::Enabled)
    .build::<EngineEvent, _>()
    .expect("failed to build system")
    .init_with_runtime(tokio::runtime::Handle::current())
    .await
    .expect("failed to init system");

println!("System built and running!");

// Let it process (with our empty stream, it will finish quickly)
tokio::time::sleep(Duration::from_secs(1)).await;

// Graceful shutdown sequence
system.cancel_orders(InstrumentFilter::None);
system.close_positions(InstrumentFilter::None);

let (engine, _) = system.shutdown().await.expect("shutdown failed");

// Generate daily trading summary
let summary = engine
    .trading_summary_generator(dec!(0.05))
    .generate(Daily);

summary.print_summary();
println!("\nEngine shut down successfully.");

## Engine Architecture Overview

```
┌─────────────────┐     ┌──────────────────┐
│  Market Data     │────▶│                  │
│  (barter-data)   │     │     Engine        │────▶ AuditStream
│                  │     │   (Processor)      │       (optional)
└─────────────────┘     │                    │
                        │  ┌──────────────┐  │
┌─────────────────┐     │  │ EngineState   │  │
│  Account Events  │────▶│  │  - positions  │  │
│  (execution rx)  │     │  │  - balances   │  │
│                  │     │  │  - orders     │  │
└─────────────────┘     │  └──────────────┘  │
                        │                    │
┌─────────────────┐     │  ┌──────────────┐  │     ┌──────────────┐
│  Commands        │────▶│  │ Strategy     │──│────▶│  Execution   │
│  (external ctrl) │     │  │ RiskManager  │  │     │  (barter-    │
│                  │     │  └──────────────┘  │     │   execution) │
└─────────────────┘     └──────────────────┘     └──────────────┘
```

### Key Components

| Component | Trait | Purpose |
|-----------|-------|---------|
| Strategy | `AlgoStrategy` | Generate order requests from market state |
| Strategy | `ClosePositionsStrategy` | Generate orders to flatten all positions |
| Strategy | `OnDisconnectStrategy` | React to exchange disconnections |
| RiskManager | `RiskManager` | Approve/reject/modify orders before execution |
| Clock | `EngineClock` | `LiveClock` (real-time) or `HistoricalClock` (backtest) |
| Execution | `ExecutionClient` | Submit orders to exchange (mock or live) |